# Full Pipeline - VAE Healer + ResNet Expert
Integrate trained Stage A and Stage B modules into one self-healing inference pipeline.

In [ ]:
import time
import yaml
import torch
import matplotlib.pyplot as plt

from src.dataset import get_dataloaders, NoiseInjector
from src.conv_vae import ConvVAE
from src.classifier import get_classifier
from src.pipeline import SelfHealingPipeline

In [ ]:
with open('../configs/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

device = config['training']['device'] if torch.cuda.is_available() else 'cpu'
vae = ConvVAE(latent_dim=config['vae']['latent_dim']).to(device)
classifier = get_classifier(num_classes=config['dataset']['num_classes'], pretrained=False).to(device)

vae.load_state_dict(torch.load('../models/conv_vae_best.pth', map_location=device))
classifier.load_state_dict(torch.load('../models/resnet_classifier.pth', map_location=device))
vae.eval()
classifier.eval()

In [ ]:
pipeline = SelfHealingPipeline(vae=vae, classifier=classifier).to(device)
pipeline

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    root=config['dataset']['root'],
    image_size=config['dataset']['image_size'],
    batch_size=8,
    train_split=config['dataset']['train_split'],
    num_workers=config['training']['num_workers'],
    noisy=False
)

injector = NoiseInjector()
images, labels = next(iter(test_loader))
images = images[:5].to(device)
noisy = torch.stack([injector.add_gaussian_noise(img, std=0.5) for img in images])
preds, confs, healed = pipeline(noisy)

fig, axes = plt.subplots(5, 4, figsize=(14, 16))
for i in range(5):
    axes[i, 0].imshow(images[i].detach().cpu().permute(1, 2, 0).numpy(), vmin=-2, vmax=2)
    axes[i, 0].set_title('Input')
    axes[i, 1].imshow(noisy[i].detach().cpu().permute(1, 2, 0).numpy(), vmin=-2, vmax=2)
    axes[i, 1].set_title('Noisy')
    axes[i, 2].imshow(healed[i].detach().cpu().permute(1, 2, 0).numpy())
    axes[i, 2].set_title('Healed')
    axes[i, 3].text(0.05, 0.5, f'Pred: {int(preds[i])}\nConf: {float(confs[i]):.3f}', fontsize=11)
    axes[i, 3].set_title('Prediction')
    for j in range(4):
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
warmup = noisy[:1]
_ = pipeline(warmup)

runs = 30
start = time.perf_counter()
for _ in range(runs):
    _ = pipeline(warmup)
elapsed_ms = (time.perf_counter() - start) * 1000 / runs
print(f'Average end-to-end inference time: {elapsed_ms:.2f} ms / sample')

In [ ]:
batch = noisy
with torch.no_grad():
    pred_b, conf_b, healed_b = pipeline(batch)

print('Batch size:', batch.size(0))
print('Predictions:', pred_b.tolist())
print('Confidences:', [round(float(c), 4) for c in conf_b])
print('Healed batch shape:', tuple(healed_b.shape))

## Summary
The integrated pipeline successfully denoises corrupted inputs and produces classifier predictions with measurable end-to-end latency.